In [26]:
import pandas as pd
import os

Load the DataFrames and add "season", "competition" columns

In [27]:
# Function to load league data from a given path and competition name
def load_league(path, competition_name):
    dfs = []

    for file in os.listdir(path):
        if file.endswith('.csv'):
            df = pd.read_csv(os.path.join(path, file))

            # season parse
            # ex: serie_a_20-21.csv
            season_part = file.split('_')[1]
            start, end = season_part.split('-')
            season = f"20{start}-20{end}"

            df['season'] = season
            df['competition'] = competition_name

            dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)


# Load all leagues
serie_a = load_league("../data/raw/serie-a", "Serie_A")
premier_league = load_league("../data/raw/premier-league", "Premier_League")
bundesliga = load_league("../data/raw/bundesliga", "Bundesliga")
la_liga = load_league("../data/raw/la-liga", "La_Liga")
ligue_1 = load_league("../data/raw/ligue-1", "Ligue_1")
super_lig = load_league("../data/raw/super-lig", "Super_Lig")
champions_league = load_league("../data/raw/champions-league", "Champions_League")
print(serie_a.head())

   Match Number  Round Number              Date               Location  \
0             1             1  19/09/2020 18:00        Artemio Franchi   
1             2             1  19/09/2020 20:45  Marcantonio Bentegodi   
2             3             1  20/09/2020 12:30          Ennio Tardini   
3             4             1  20/09/2020 15:00         Luigi Ferraris   
4             5             1  20/09/2020 18:00          Mapei Stadium   

       Home Team Away Team Result     season competition  
0     Fiorentina    Torino  1 - 0  2020-2021     Serie_A  
1  Hellas Verona      Roma  0 - 0  2020-2021     Serie_A  
2          Parma    Napoli  0 - 2  2020-2021     Serie_A  
3          Genoa   Crotone  4 - 1  2020-2021     Serie_A  
4       Sassuolo  Cagliari  1 - 1  2020-2021     Serie_A  


Renaming columns

In [28]:
dfs = [serie_a, premier_league, bundesliga, la_liga, ligue_1, super_lig, champions_league]

# Standardize column names
rename_dict = {
    "Home Team": "target_team",
    "Away Team": "opponent_team",
    "Date": "date",
    "Match Number": "match_number",
    "Round Number": "stage",
    "Result": "result"
}

# Rename columns and drop unnecessary ones
for df in dfs:
    df.rename(columns=rename_dict, inplace=True)
    df.drop(columns=['Location', 'Group'], inplace=True, errors='ignore')

# Convert stage to string
for df in dfs:
    df["stage"] = df["stage"].astype(str)

print(serie_a.head())

   match_number stage              date    target_team opponent_team result  \
0             1     1  19/09/2020 18:00     Fiorentina        Torino  1 - 0   
1             2     1  19/09/2020 20:45  Hellas Verona          Roma  0 - 0   
2             3     1  20/09/2020 12:30          Parma        Napoli  0 - 2   
3             4     1  20/09/2020 15:00          Genoa       Crotone  4 - 1   
4             5     1  20/09/2020 18:00       Sassuolo      Cagliari  1 - 1   

      season competition  
0  2020-2021     Serie_A  
1  2020-2021     Serie_A  
2  2020-2021     Serie_A  
3  2020-2021     Serie_A  
4  2020-2021     Serie_A  


Normalizing and adding columns

In [29]:
final_columns = [
    'target_team',
    'opponent_team',
    'is_home',
    'date',
    'competition',
    'stage',
    'season',
    'days_since_last_ucl_match',
    'ucl_impact_category',
    'result',
    'target_team_goals',
    'opponent_team_goals',
    'points',
    'ucl_format',
    'target_team_match_count',
    'opponent_team_match_count',
]

def prepare_dataframe(df):
    for col in final_columns:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[final_columns]
    return df

dfs = [prepare_dataframe(df) for df in dfs]
serie_a, premier_league, bundesliga, la_liga, ligue_1, super_lig, champions_league = dfs

print(serie_a.columns)
print(serie_a.head())

Index(['target_team', 'opponent_team', 'is_home', 'date', 'competition',
       'stage', 'season', 'days_since_last_ucl_match', 'ucl_impact_category',
       'result', 'target_team_goals', 'opponent_team_goals', 'points',
       'ucl_format', 'target_team_match_count', 'opponent_team_match_count'],
      dtype='object')
     target_team opponent_team is_home              date competition stage  \
0     Fiorentina        Torino    <NA>  19/09/2020 18:00     Serie_A     1   
1  Hellas Verona          Roma    <NA>  19/09/2020 20:45     Serie_A     1   
2          Parma        Napoli    <NA>  20/09/2020 12:30     Serie_A     1   
3          Genoa       Crotone    <NA>  20/09/2020 15:00     Serie_A     1   
4       Sassuolo      Cagliari    <NA>  20/09/2020 18:00     Serie_A     1   

      season days_since_last_ucl_match ucl_impact_category result  \
0  2020-2021                      <NA>                <NA>  1 - 0   
1  2020-2021                      <NA>                <NA>  0 - 0   
2 

Merging the Sesons and load as CSV

In [30]:
all_leagues = pd.concat([serie_a, premier_league, bundesliga, la_liga, ligue_1, super_lig, champions_league], ignore_index=True)
print(all_leagues.head())

     target_team opponent_team is_home              date competition stage  \
0     Fiorentina        Torino     NaN  19/09/2020 18:00     Serie_A     1   
1  Hellas Verona          Roma     NaN  19/09/2020 20:45     Serie_A     1   
2          Parma        Napoli     NaN  20/09/2020 12:30     Serie_A     1   
3          Genoa       Crotone     NaN  20/09/2020 15:00     Serie_A     1   
4       Sassuolo      Cagliari     NaN  20/09/2020 18:00     Serie_A     1   

      season days_since_last_ucl_match ucl_impact_category result  \
0  2020-2021                       NaN                 NaN  1 - 0   
1  2020-2021                       NaN                 NaN  0 - 0   
2  2020-2021                       NaN                 NaN  0 - 2   
3  2020-2021                       NaN                 NaN  4 - 1   
4  2020-2021                       NaN                 NaN  1 - 1   

  target_team_goals opponent_team_goals points ucl_format  \
0               NaN                 NaN    NaN        N

Tranform the data to team centric format and load the data as csv

In [31]:
# load the all_teams_raw_normalized.csv file
all_leagues_normalized = pd.read_csv("../data/processed/all_matches_raw_normalized.csv")

# perspective 1: Home team is the target team
# the original 'target_team' in raw data is the home team
home_perspective_data = all_leagues_normalized.copy()
home_perspective_data['is_home'] = True

# perspective 2: Away team is the target team
# we swap the theams and their respective goals
away_perspective_data = all_leagues_normalized.copy()
away_perspective_data['target_team'] = all_leagues_normalized['opponent_team']
away_perspective_data['opponent_team'] = all_leagues_normalized['target_team']
away_perspective_data['target_team_goals'] = all_leagues_normalized['opponent_team_goals']
away_perspective_data['opponent_team_goals'] = all_leagues_normalized['target_team_goals']
away_perspective_data['is_home'] = False

# Combine the home and away perspectives into one master long-format DataFrame
team_centric_matches = pd.concat([home_perspective_data, away_perspective_data], ignore_index=True)

# sort the data cronologically for each team
# This sorting is CRITICAL for calculating match counts and previous match status later
# date to datetime
team_centric_matches['date'] = pd.to_datetime(team_centric_matches['date'], dayfirst=True)
team_centric_matches = team_centric_matches.sort_values(by=['target_team', 'season', 'date']).reset_index(drop=True)

# Verification: check the results for a specific team
print(team_centric_matches[team_centric_matches['target_team'] == 'Galatasaray'].head())

# load the data as csv
team_centric_matches.to_csv("../data/processed/team_centric_matches.csv", index=False)

       target_team        opponent_team  is_home                date  \
10189  Galatasaray       Gaziantep F.K.     True 2020-09-12 20:00:00   
10190  Galatasaray  Istanbul Basaksehir    False 2020-09-20 19:00:00   
10191  Galatasaray           Fenerbahçe     True 2020-09-27 19:00:00   
10192  Galatasaray            Kasimpasa    False 2020-10-04 19:00:00   
10193  Galatasaray           Alanyaspor     True 2020-10-19 20:00:00   

      competition stage     season  days_since_last_ucl_match  \
10189   Super_Lig     1  2020-2021                        NaN   
10190   Super_Lig     2  2020-2021                        NaN   
10191   Super_Lig     3  2020-2021                        NaN   
10192   Super_Lig     4  2020-2021                        NaN   
10193   Super_Lig     5  2020-2021                        NaN   

       ucl_impact_category result  target_team_goals  opponent_team_goals  \
10189                  NaN  3 - 1                NaN                  NaN   
10190                 